# 08 - EAD & CCF (Exposure at Default) + Expected Loss

**Plain-English summary (read this first)**

When a borrower stops paying, the bank loses money on whatever they still owe at that moment.
That amount is called **EAD - Exposure at Default** (simply: the balance owed when the account goes bad).
This notebook measures EAD on the **credit-card** accounts in the data, and a related number
called the **CCF - Credit Conversion Factor** (how much of a customer's *unused* card limit tends
to get drawn down on the way into default). It then combines EAD with the default probability (PD)
from the existing scorecard and a standard loss assumption (LGD) to estimate **Expected Loss = PD x LGD x EAD**.

**Headline result (from the latest run):** about **1,495** credit-card accounts defaulted with usable history.
Their **average EAD is ~3,000** (currency units of the data) and the **average CCF is low (~1%)** - in this
particular dataset most card balances were already paid down to a small residual *before* the 90-day-late mark,
so very little extra was drawn. A worked Expected-Loss example is shown near the bottom, and the one-line results
table is saved to `output/ead_summary.csv`.

> **Important - read the "Data-quality finding" section near the end.** A sanity check showed the 90+ days-past-due
> flag on this dataset **does not represent economic default** (`SK_DPD` accumulates and never resets, so the flag
> fires late on a tiny residual, and some accounts even revive afterwards). Because of that, the EAD/CCF/EL numbers
> here are a **methodology demonstration, not a credible loss estimate**. Real, properly-anchored EAD is demonstrated
> in the companion **Freddie Mac mortgage project**, which has a clean default definition. An **onset-of-delinquency**
> cross-check is included below and confirms the exposure is genuinely small.


## What this notebook does, step by step

1. Read the credit-card monthly file and keep only the columns we need.
2. Mark an account as **defaulted** when it is **90+ days past due** (`SK_DPD >= 90`) - the standard Basel definition.
3. For each defaulted account, find the **default month** (first month it crosses 90 days late) and look **12 months earlier** - the **reference point**.
4. **EAD** = the balance in the default month. **CCF** = how much of the unused limit at the reference point got used by default.
5. Summarise: average CCF, CCF split by how much of the limit was used (low / medium / high), and a predicted-EAD formula.
6. Add an **Expected Loss** section: `EL = PD x LGD x EAD`, using a PD from the scorecard and a clearly-labelled LGD assumption.

**Columns used (the real names in `data/raw/credit_card_balance.csv`):**

| Meaning | Column |
|---|---|
| Account id | `SK_ID_PREV` |
| Client id | `SK_ID_CURR` |
| Month index (-1 = freshest, more negative = older) | `MONTHS_BALANCE` |
| Balance owed that month | `AMT_BALANCE` |
| Credit limit that month | `AMT_CREDIT_LIMIT_ACTUAL` |
| Total drawings that month | `AMT_DRAWINGS_CURRENT` |
| Days past due that month | `SK_DPD` |


In [1]:
# Load libraries and locate the repo root so paths work whether run from the repo or the notebooks folder.
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "credit_card_balance.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent   # walk up until we find the data folder

DATA = ROOT / "data" / "raw" / "credit_card_balance.csv"
PD_OUTPUT = ROOT / "output" / "scorecard_outputs" / "06_test_scored.csv"   # existing PD scorecard result
OUT_DIR = ROOT / "output"; OUT_DIR.mkdir(exist_ok=True)   # existing output/ folder
print("Repo root:", ROOT)
print("Reading from:", DATA)

Repo root: D:\Jane\Job Search\Github\bank\github project\scorecard pd ead consummer credit
Reading from: D:\Jane\Job Search\Github\bank\github project\scorecard pd ead consummer credit\data\raw\credit_card_balance.csv


In [2]:
# Read only the seven columns we need from the (large) credit-card balance file.
cols = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE",
        "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT", "SK_DPD"]
cc = pd.read_csv(DATA, usecols=cols)
print(f"{len(cc):,} monthly rows across {cc.SK_ID_PREV.nunique():,} credit-card accounts")
cc.head()

3,840,312 monthly rows across 104,307 credit-card accounts


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_CURRENT,SK_DPD
0,2562384,378907,-6,56.970,135000,877.5,0
1,2582071,363914,-1,63975.555,45000,2250.0,0
2,1740877,371185,-7,31815.225,450000,0.0,0
3,1389973,337855,-4,236572.110,225000,2250.0,0
4,1891521,126868,-1,453919.455,450000,11547.0,0


In [3]:
# Flag every 90+ days-past-due month and find each account's first (earliest) default month.
defaulted_rows = cc[cc.SK_DPD >= 90]
default_month = (defaulted_rows
                 .groupby("SK_ID_PREV")["MONTHS_BALANCE"].min()   # most negative = earliest in time
                 .rename("def_month").reset_index())
print(f"{len(default_month):,} accounts ever reached 90+ days past due (defaulted)")

1,806 accounts ever reached 90+ days past due (defaulted)


In [4]:
# EAD = the balance in the default month, for each defaulted account.
ead = (cc.merge(default_month, on="SK_ID_PREV")
         .query("MONTHS_BALANCE == def_month")
         .groupby("SK_ID_PREV", as_index=False)
         .agg(def_month=("def_month", "first"), EAD=("AMT_BALANCE", "max")))
print(f"EAD measured for {len(ead):,} accounts")

EAD measured for 1,806 accounts


In [5]:
# Look 12 months before each default (the reference point) and record balance and limit there.
ead["ref_month"] = ead["def_month"] - 12   # 12 months earlier = subtract 12 from the month index
ref = (cc.merge(ead[["SK_ID_PREV", "ref_month"]], on="SK_ID_PREV")
         .query("MONTHS_BALANCE == ref_month")
         .groupby("SK_ID_PREV", as_index=False)
         .agg(bal_ref=("AMT_BALANCE", "first"), limit_ref=("AMT_CREDIT_LIMIT_ACTUAL", "first")))

df = ead.merge(ref, on="SK_ID_PREV")
print(f"{len(df):,} accounts have a balance/limit record 12 months before default")

1,595 accounts have a balance/limit record 12 months before default


In [6]:
# Compute the CCF, keeping only accounts that had spare limit (headroom) at the reference point.
df = df[df.limit_ref > df.bal_ref].copy()          # need positive headroom or CCF is undefined
df["headroom_ref"] = df.limit_ref - df.bal_ref
df["CCF"] = ((df.EAD - df.bal_ref) / df.headroom_ref).clip(0, 1)   # capped between 0 and 1
df["util_ref"] = (df.bal_ref / df.limit_ref)        # how much of the limit was used at reference
print(f"{len(df):,} defaulted accounts used for EAD/CCF")
print("Average CCF: {:.3f}".format(df.CCF.mean()))
print("Average EAD: {:,.0f}".format(df.EAD.mean()))

1,495 defaulted accounts used for EAD/CCF
Average CCF: 0.009
Average EAD: 2,937


In [7]:
# Show average CCF split by how much of the limit was used at the reference point (low / medium / high).
bands = pd.cut(df.util_ref, [-0.001, 0.30, 0.70, np.inf], labels=["low", "medium", "high"])
ccf_by_band = (df.groupby(bands, observed=True)["CCF"]
                 .agg(accounts="count", avg_CCF="mean").round(3))
ccf_by_band

,accounts,avg_CCF
util_ref,,
low,705,0.007
medium,501,0.005
high,289,0.022


In [8]:
# Apply the predicted-EAD formula to one example account: balance + CCF x (limit - balance).
avg_ccf = float(df.CCF.mean())
example = df.iloc[0]
predicted_ead = example.bal_ref + avg_ccf * (example.limit_ref - example.bal_ref)
print("Predicted-EAD formula:  EAD = current_balance + CCF x (limit - current_balance)")
print(f"Example account {int(example.SK_ID_PREV)}:")
print(f"  balance={example.bal_ref:,.0f}  limit={example.limit_ref:,.0f}  CCF(avg)={avg_ccf:.3f}")
print(f"  predicted EAD = {predicted_ead:,.0f}")

Predicted-EAD formula:  EAD = current_balance + CCF x (limit - current_balance)
Example account 1000320:
  balance=135  limit=112,500  CCF(avg)=0.009
  predicted EAD = 1,177


**Why we build EAD/CCF on defaulted accounts only:** EAD answers "how much was owed *at the moment of default*",
so it can only be measured on accounts that actually reached default - there is no default month for accounts that never went bad.
This is the correct, standard approach: the resulting CCF is then applied to the accounts the PD model *predicts* will default.

## Expected Loss (EL = PD x LGD x EAD)

**EL** is the average loss we expect from an account, combining three pieces:
- **PD - Probability of Default:** chance the account goes bad (taken from the existing scorecard).
- **LGD - Loss Given Default:** share of the exposure we *don't* recover (assumed here - see label below).
- **EAD - Exposure at Default:** the amount owed when it goes bad (measured above).


In [9]:
# Pull a PD from the existing scorecard, set an assumed LGD, and work through PD x LGD x EAD = EL.
pd_book = pd.read_csv(PD_OUTPUT, usecols=["pd_pred"])
PD = float(pd_book.pd_pred.mean())            # portfolio-average predicted PD from the scorecard
LGD = 0.70                                    # ASSUMPTION - no recovery data in this dataset (unsecured consumer)
EAD_example = float(df.EAD.mean())            # average measured EAD on the credit-card segment

EL = PD * LGD * EAD_example
print("ASSUMPTION - LGD fixed at 0.70: this dataset has no recovery data, so loss-given-default is assumed.")
print(f"Worked example:  PD={PD:.4f}  x  LGD={LGD:.2f}  x  EAD={EAD_example:,.0f}  =  EL={EL:,.2f}")

ASSUMPTION - LGD fixed at 0.70: this dataset has no recovery data, so loss-given-default is assumed.
Worked example:  PD=0.0805  x  LGD=0.70  x  EAD=2,937  =  EL=165.40


### Data note - why the CCF is low here

In this dataset `SK_DPD` (days past due) **accumulates** once an account turns delinquent - it keeps counting up and
never resets. In practice, balances are usually paid down to a small residual *before* the 90-day mark is crossed, so
the balance at the official default month is often tiny and frequently *lower* than 12 months earlier. The CCF therefore
clips to 0 for most accounts and the average comes out low (~1%). This is a property of how the data records delinquency,
not a calculation error. On a cleaner book (where delinquency and balances are observed at the true point of default),
the same code would produce the usual 30-75% credit-card CCF range.

## EAD at onset of delinquency (cross-check on the headline)

Instead of the balance at the 90-day mark, measure the balance in the **first month the account
became delinquent at all** (the start of the late streak that ends in default). This is the closest
this dataset gets to a true "exposure at default" point - the exposure *as the account starts to go bad*,
before months of accumulating days-past-due drag it to the 90-day flag.

It comes out **also small (~2,200 on average)**, consistent with the **~2,937 balance-at-default headline**.
Both say the same thing: by the time these accounts turn delinquent, the balance is already a small residual.
The balance-at-default figure stays the **primary** number; onset is shown as a cross-check.

In [10]:
# EAD at onset of delinquency = the balance in the first month of the late streak that ends in default.
# (Walk back from the default month while the account is still past due; the start of that run is the onset.)
sub = (cc.merge(df[["SK_ID_PREV", "def_month"]], on="SK_ID_PREV")
         .query("MONTHS_BALANCE <= def_month")
         .sort_values(["SK_ID_PREV", "MONTHS_BALANCE"]))

def _onset(g):
    dpd, m, b = g.SK_DPD.values, g.MONTHS_BALANCE.values, g.AMT_BALANCE.values
    i = len(g) - 1                       # last row in the window = the default month
    while i > 0 and dpd[i - 1] > 0:      # step back while the prior month is also past due
        i -= 1
    return pd.Series({"onset_month": m[i], "EAD_onset": b[i]})

onset = (sub.groupby("SK_ID_PREV")[["MONTHS_BALANCE", "AMT_BALANCE", "SK_DPD"]]
            .apply(_onset).reset_index())
df = df.merge(onset, on="SK_ID_PREV")

print("Average EAD (onset of delinquency):        {:,.0f}".format(df.EAD_onset.mean()))
print("Average EAD (balance at default, headline): {:,.0f}".format(df.EAD.mean()))
print("-> the onset figure is also small, consistent with the balance-at-default headline:")
print("   by the time these accounts turn delinquent, the balance is already a small residual.")

Average EAD (onset of delinquency):        2,228
Average EAD (balance at default, headline): 2,937
-> the onset figure is also small, consistent with the balance-at-default headline:
   by the time these accounts turn delinquent, the balance is already a small residual.


## Data-quality finding (the real takeaway of this notebook)

Treat this notebook as a **data-quality finding, not an EAD model**. A sanity check on the
exposure numbers showed that the default flag itself is not trustworthy on this dataset.

- **The 90+ days-past-due flag here does not represent economic default.** It is a mechanical
  delinquency counter, not the moment a borrower actually went bad.
- **Evidence (see the diagnostic cell and traced accounts below):**
  - The largest balance in the year before "default" lands, on average, **~10 months earlier**,
    in a month when the account was **fully current (DPD = 0)** - normal revolving usage, not a run-up into default.
  - The 90-day flag then fires **months later on a tiny stranded residual** (often a few units of currency).
  - Some accounts **revive after their "default" month**, with balances climbing back to ~100k-200k for
    years afterwards - so the flagged month was never a real default at all.
- **Cause:** `SK_DPD` (days past due) **accumulates and never resets**. Once an account is briefly late, the
  counter keeps ticking up on whatever residual remains until it crosses 90 - regardless of whether the borrower
  has actually defaulted.
- **Conclusion:** this dataset **cannot support a credible revolving EAD / CCF model**. A correct approach would
  anchor exposure at the **onset of delinquency** (done above as a cross-check) and, crucially, use a dataset with a
  **clean, non-reviving default definition**.
- **Where EAD is done properly:** exposure-at-default is demonstrated correctly in the companion
  **Freddie Mac mortgage project**, where the default definition is clean (terminations/defaults are observed
  unambiguously and do not revive).

The headline EAD/CCF/EL numbers above are kept only as a **methodology demonstration** of the mechanics
(`EL = PD x LGD x EAD`); they are **not credible loss estimates** for this book.


In [11]:
# EVIDENCE (diagnostic only - NOT used as an EAD metric):
# where does the largest balance in the 12 months up to default actually sit?
win = cc.merge(df[["SK_ID_PREV", "def_month"]], on="SK_ID_PREV")
win = win[(win.MONTHS_BALANCE >= win.def_month - 12) & (win.MONTHS_BALANCE <= win.def_month)]
pk_idx = win.groupby("SK_ID_PREV")["AMT_BALANCE"].idxmax()
pk = (win.loc[pk_idx, ["SK_ID_PREV", "MONTHS_BALANCE", "AMT_BALANCE"]]
        .rename(columns={"MONTHS_BALANCE": "peak_month", "AMT_BALANCE": "peak_bal"}))
ev = df.merge(pk, on="SK_ID_PREV")

evidence = pd.DataFrame([
    {"check": "peak balance lands IN the default month",     "value": "{:.1%}".format((ev.peak_month == ev.def_month).mean())},
    {"check": "peak balance lands AFTER default (leak)",     "value": str(int((ev.peak_month > ev.def_month).sum()))},
    {"check": "avg months peak sits BEFORE default",         "value": "{:.1f}".format((ev.def_month - ev.peak_month).mean())},
    {"check": "avg peak balance (a healthy-period high)",    "value": "{:,.0f}".format(ev.peak_bal.mean())},
    {"check": "avg balance at default (headline EAD)",       "value": "{:,.0f}".format(ev.EAD.mean())},
    {"check": "median peak / balance-at-default ratio",      "value": "{:.0f}x".format((ev.peak_bal / ev.EAD.replace(0, np.nan)).median())},
])
print("The 12-month peak balance is NOT exposure at default: it sits ~10 months earlier, in a")
print("fully-current period (DPD = 0). Shown here only as evidence of the data-quality problem.\n")
print(evidence.to_string(index=False))


# Two traced accounts: month-by-month balance / limit / days-past-due paths.
def trace(pid, note):
    r = ev[ev.SK_ID_PREV == pid].iloc[0]
    path = cc[cc.SK_ID_PREV == pid].sort_values("MONTHS_BALANCE")
    path = path[path.MONTHS_BALANCE >= int(r.def_month) - 13]   # window + any post-default months
    print(f"\nAccount {pid} - {note}")
    print(f"  def_month={int(r.def_month)}  peak_month={int(r.peak_month)}  "
          f"balance@default(EAD)={r.EAD:,.0f}  peak_balance={r.peak_bal:,.0f}")
    print(f"  {'month':>6}{'balance':>13}{'limit':>11}{'DPD':>6}   note")
    for _, m in path.iterrows():
        tag = []
        if m.MONTHS_BALANCE == int(r.peak_month): tag.append("<- PEAK (DPD=0, healthy)")
        if m.MONTHS_BALANCE == int(r.def_month):  tag.append("<- DEFAULT (90+ DPD)")
        if m.MONTHS_BALANCE > int(r.def_month):   tag.append("post-default")
        print(f"  {int(m.MONTHS_BALANCE):>6}{m.AMT_BALANCE:>13,.0f}{m.AMT_CREDIT_LIMIT_ACTUAL:>11,.0f}{int(m.SK_DPD):>6}   {' '.join(tag)}")

trace(1126779, "peaks at ~478k while fully current, then defaults months later on a residual of ~9")
trace(2235755, "REVIVES after its 'default' month - balances climb back to ~100k-196k for two more years")

The 12-month peak balance is NOT exposure at default: it sits ~10 months earlier, in a
fully-current period (DPD = 0). Shown here only as evidence of the data-quality problem.

                                   check  value
 peak balance lands IN the default month   3.1%
 peak balance lands AFTER default (leak)      0
     avg months peak sits BEFORE default   10.5
avg peak balance (a healthy-period high) 55,448
   avg balance at default (headline EAD)  2,937
  median peak / balance-at-default ratio   178x

Account 1126779 - peaks at ~478k while fully current, then defaults months later on a residual of ~9
  def_month=-2  peak_month=-9  balance@default(EAD)=9  peak_balance=477,971
   month      balance      limit   DPD   note
     -15      185,756    450,000     0   
     -14      273,331    450,000     0   
     -13      346,209    450,000     0   
     -12      374,650    450,000     0   
     -11      461,407    450,000     0   
     -10      425,124    450,000     0   
      -9   

In [12]:
# Build one clean results table (CCF, EAD, EL example) and save it to output/ead_summary.csv.
# NOTE: methodology-demonstration figures only - see the "Data-quality finding" section above.
summary = pd.DataFrame([
    {"metric": "defaulted_accounts_used", "value": round(len(df), 0)},
    {"metric": "average_CCF",             "value": round(df.CCF.mean(), 4)},
    {"metric": "average_EAD",             "value": round(df.EAD.mean(), 2)},
    {"metric": "average_EAD_onset",       "value": round(df.EAD_onset.mean(), 2)},
    {"metric": "CCF_low_util",            "value": round(df.loc[df.util_ref <= 0.30, "CCF"].mean(), 4)},
    {"metric": "CCF_medium_util",         "value": round(df.loc[(df.util_ref > 0.30) & (df.util_ref <= 0.70), "CCF"].mean(), 4)},
    {"metric": "CCF_high_util",           "value": round(df.loc[df.util_ref > 0.70, "CCF"].mean(), 4)},
    {"metric": "PD_from_scorecard",       "value": round(PD, 4)},
    {"metric": "LGD_assumed",             "value": LGD},
    {"metric": "EAD_example",             "value": round(EAD_example, 2)},
    {"metric": "EL_example_PDxLGDxEAD",   "value": round(EL, 2)},
])
summary.to_csv(OUT_DIR / "ead_summary.csv", index=False)
print("Saved -> output/ead_summary.csv  (methodology-demonstration figures, not credible loss estimates)")
summary

Saved -> output/ead_summary.csv  (methodology-demonstration figures, not credible loss estimates)


,metric,value
0,defaulted_accounts_used,1495.0000
1,average_CCF,0.0093
2,average_EAD,2936.9900
3,average_EAD_onset,2228.2600
4,CCF_low_util,0.0073
5,CCF_medium_util,0.0050
6,CCF_high_util,0.0215
7,PD_from_scorecard,0.0805
8,LGD_assumed,0.7000
9,EAD_example,2936.9900
